# Experimento D2 · Partición por año y partition pruning
**Responsable:** Andrade y Carranza

## 1. Objetivo
Medir cuánto reduce el tiempo de consulta que la tabla esté particionada por año cuando se filtra por año.

**Hipótesis:** Con la tabla particionada, filtrar por un año lee solo esa carpeta y es más rápido que con la tabla sin particionar.

## 2. Configuración y condiciones de prueba
- Escala 5x (5,048,575 filas), con una copia particionada por `anio` y otra sin particionar.
- Filtros: sin filtro, un año (2019) y un rango (2015-2017).
- Consulta: KPIs por año con `collect()`.

## 3. Procedimiento
1. Ejecuta `%run ./00_utilidades`.
2. Ejecuta las celdas de medición en orden. Cada medida hace **1 ejecución de calentamiento (no se cuenta) y 3 repeticiones**.
3. Los resultados se guardan solos en la tabla `resultados_experimentos`.
4. Completa el análisis (sección 5) con tus números.

**Evidencias a capturar:** el plan de ejecución (`explain`) con `PartitionFilters`, la medición y la tabla de resultados.

In [ ]:
%run ./00_utilidades

In [ ]:
EXP, RESP = "D2_particionado", "Andrade y Carranza"
guardar_tabla(leer_tabla(T_SILVER_5X), "silver_5x_sin_particion")      # copia SIN particionar
con = leer_tabla(T_SILVER_5X)                                          # particionada por anio
sin = leer_tabla("silver_5x_sin_particion")
print(f"Filas: {con.count():,} (particionada) | {sin.count():,} (sin particionar)")

### Evidencia del pruning
En el plan de la tabla particionada debe aparecer `PartitionFilters`. En la otra, solo `PushedFilters`.

In [ ]:
con.where("anio = 2019").explain()
print("Archivos leídos (particionada):", len(con.where("anio = 2019").inputFiles()), "de", len(con.inputFiles()))
print("Archivos leídos (sin particionar):", len(sin.where("anio = 2019").inputFiles()), "de", len(sin.inputFiles()))

## 4. Resultados

In [ ]:
casos = [("sin_filtro", None), ("anio_2019", "anio = 2019"), ("anios_2015_2017", "anio BETWEEN 2015 AND 2017")]
for nombre, filtro in casos:
    for etiqueta, tabla in [("particionada", con), ("sin_particion", sin)]:
        d = tabla.where(filtro) if filtro else tabla
        filas = d.count()
        tiempos = medir(lambda x=d: calcular_kpis(x, "anio").collect())
        f = registrar_resultado(EXP, f"{etiqueta}_{nombre}", "5x", filas, tiempos, "Spark", RESP,
                                f"filtro: {filtro or 'ninguno'}")
        print(f"{etiqueta:14s} {nombre:16s} {filas:>10,} filas | {f.media_s} s (± {f.desv_s})")

In [ ]:
display(leer_tabla(T_RESULTADOS).where("experimento = 'D2_particionado'").orderBy("variante"))

## 5. Comparación y análisis (completa con tus números)
- Con el filtro de un año, ¿cuántas veces más rápida fue la tabla particionada?
- Sin filtro, ¿hubo diferencia? ¿Por qué?
- ¿Cuántos archivos leyó Spark en cada caso?
- Conclusión: ¿vale la pena particionar por año?